# 🧫 19 · Deep Learning Segmentation
### From fully convolutional networks to U-Net: giving every pixel its own label, learned end-to-end

**Part F · Segmenting, detecting, synthesizing** · Module 19 of 37 (core) · builds on 16, 18

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xulabs/edu/blob/main/sci_img/tutorial/BiologyToBioimage/notebooks/19_segmentation_deep.ipynb)

| | |
|---|---|
| **Hardware target** | CPU (a GPU runtime speeds up Section 6, but is not required) |
| **Core frameworks** | NumPy, PyTorch |
| **Prerequisites** | Module 16 (CNN basics), Module 18 (classical segmentation) |
| **Estimated time** | 100–130 minutes |
| **02-740 topics covered** | semantic/instance/panoptic segmentation, fully convolutional networks, transposed convolution, the U-Net architecture, weighted loss for touching objects, V-Net and the Dice loss, flow-based (Cellpose-style) instance segmentation |

> ⚠️ **Important notes.** For research and educational purposes only, not to be used for clinical diagnosis.
> Every training image and its instance mask are generated by `course_toolkit.simulate_nuclei_image`, so Dice/IoU scores below are computed against an exact, known-correct segmentation.

## 🎯 Learning objectives

By the end of this notebook you will be able to:

- Distinguish semantic, instance and panoptic segmentation.
- Explain how a fully convolutional network (FCN) turns a classification network into a dense,
  pixel-wise predictor, and what transposed convolution does.
- Explain the U-Net architecture and precisely what its skip connections add beyond a plain FCN.
- Explain why a weighted loss helps separate touching objects of the same class, and build the
  corresponding weight map from a known instance mask.
- Explain the Dice loss, why it suits small-object segmentation better than per-pixel cross-entropy, and
  train a small U-Net with it on simulated, touching nuclei.
- Explain the flow-based (Cellpose-style) alternative to loss-based instance separation — predicting a
  per-pixel vector field toward each object's centre and decoding instances by following it — and why it
  can split touching objects that a thresholded mask's connected components structurally cannot.


## 🗺️ Contents

1. Semantic, instance and panoptic segmentation
2. From classification to dense prediction: fully convolutional networks
3. Transposed convolution: upsampling that is also learned
4. U-Net: an encoder-decoder with skip connections
5. The weighted loss for touching objects
6. Dice loss and training a small U-Net
7. Flow-based instance segmentation (Cellpose-style)


## 🧫 The biological question

Module 18's classical segmentation toolbox works well for isolated, high-contrast objects but needs
substantial hand-tuning (thresholds, marker distances, snake energies) and can still fail badly on
touching, low-contrast, or highly variable real microscopy data. A learned segmentation model instead
discovers, from examples, what "this pixel belongs to a cell" and "this pixel is the boundary between two
touching cells" actually look like — the same shift from hand-crafted to learned features that Module 16
made for classification, now applied pixel-by-pixel.

---

## Environment setup

In [ ]:
# ── Hardware check ──
import platform
print('Python', platform.python_version())
try:
    import torch
    print('Torch', torch.__version__, '| CUDA available:', torch.cuda.is_available())
except ImportError:
    print('PyTorch not installed yet — the next cell installs it if needed.')


In [ ]:
!pip install -q scikit-image torch
# ── Dependencies ── (Colab already has most of these; we install only what is missing)


In [ ]:
# ── Global imports and configuration ──
import numpy as np
import matplotlib.pyplot as plt
from scipy import ndimage as ndi
import torch
import torch.nn as nn
import torch.nn.functional as F
torch.manual_seed(110)
SEED = 110
rng = np.random.default_rng(SEED)
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.titlesize'] = 11


### 🧰 Course toolkit

In [ ]:
# ── Course toolkit (identical to ../course_toolkit.py v1.0; copied so this notebook runs alone) ──
import numpy as np
import matplotlib.pyplot as plt
from scipy import ndimage as ndi
from scipy.optimize import linear_sum_assignment

SHAPE_CLASSES = ("round", "elongated", "kidney", "ring", "dividing", "irregular")


def _draw_blob(canvas: np.ndarray, cy: float, cx: float, r: float, shape: str,
                rng: np.random.Generator, angle: float = 0.0) -> None:
    """Add one synthetic nucleus/cell of the given shape class onto ``canvas`` (in place, additive)."""
    size = canvas.shape[0]
    yy, xx = np.meshgrid(np.arange(size), np.arange(size), indexing="ij")
    y, x = yy - cy, xx - cx
    ca, sa = np.cos(angle), np.sin(angle)
    yr, xr = ca * y + sa * x, -sa * y + ca * x  # rotate into the blob's own frame
    if shape == "round":
        d = (yr / r) ** 2 + (xr / r) ** 2
        canvas += np.exp(-d * 2.0)
    elif shape == "elongated":
        d = (yr / (r * 0.55)) ** 2 + (xr / (r * 1.7)) ** 2
        canvas += np.exp(-d * 2.0)
    elif shape == "kidney":
        d = (yr / r) ** 2 + (xr / r) ** 2
        dent = np.exp(-((yr - 0.0) ** 2 + (xr - r * 0.9) ** 2) / (2 * (r * 0.5) ** 2))
        canvas += np.clip(np.exp(-d * 2.0) - 0.9 * dent, 0, None)
    elif shape == "ring":
        d = np.sqrt(yr ** 2 + xr ** 2)
        canvas += np.exp(-((d - r) ** 2) / (2 * (r * 0.28) ** 2))
    elif shape == "dividing":
        sep = r * 1.1
        for sign in (-1, 1):
            d = ((yr - sign * sep * 0.5) / (r * 0.75)) ** 2 + (xr / (r * 0.75)) ** 2
            canvas += np.exp(-d * 2.0)
    else:  # irregular: a round blob perturbed by a few random bumps
        d = (yr / r) ** 2 + (xr / r) ** 2
        base = np.exp(-d * 2.0)
        for _ in range(3):
            bx, by = rng.uniform(-r, r, size=2)
            bump = np.exp(-(((yr - by) ** 2 + (xr - bx) ** 2) / (2 * (r * 0.45) ** 2)))
            base = np.clip(base + 0.5 * bump, 0, 1.3)
        canvas += base


def add_gaussian_noise(image: np.ndarray, sigma: float, rng: np.random.Generator) -> np.ndarray:
    """Additive Gaussian (read) noise: the classic noise floor of a camera sensor."""
    return image + rng.normal(0.0, sigma, size=image.shape)


def blur_image(image: np.ndarray, psf: np.ndarray) -> np.ndarray:
    """Convolve an image with a point-spread function (microscope optical blur)."""
    return ndi.convolve(image, psf, mode="reflect")


def gaussian_psf(sigma: float, radius: int | None = None) -> np.ndarray:
    """A normalised 2-D Gaussian point-spread function (the microscope's blur kernel)."""
    if radius is None:
        radius = max(1, int(np.ceil(3 * sigma)))
    ax = np.arange(-radius, radius + 1)
    xx, yy = np.meshgrid(ax, ax)
    k = np.exp(-(xx ** 2 + yy ** 2) / (2 * sigma ** 2))
    return k / k.sum()


def simulate_nuclei_image(n_cells: int = 25, size: int = 160, touching: bool = False,
                           shape_classes: tuple[str, ...] | None = None,
                           psf_sigma: float = 1.2, background: float = 0.05,
                           noise_sigma: float = 0.03, seed: int = 0):
    """A SIMULATED single-channel fluorescence image of cell nuclei/cells.

    Returns ``image`` (size x size, background + PSF-blurred blobs + Gaussian noise),
    ``instance_mask`` (0 = background, 1..n_cells = one integer id per cell, the ground truth
    for segmentation/detection/tracking), ``centroids`` (n_cells x 2, row, col), ``shape_labels``
    (one of SHAPE_CLASSES per cell, the ground truth for classification), and ``boxes`` (n_cells x 4,
    ymin, xmin, ymax, xmax — the ground truth for detection).
    """
    rng = np.random.default_rng(seed)
    classes = shape_classes or SHAPE_CLASSES
    image = np.full((size, size), background, dtype=float)
    instance_mask = np.zeros((size, size), dtype=int)
    centroids, labels, boxes = [], [], []
    margin = size * 0.12
    min_sep = size * 0.11 if not touching else size * 0.065
    placed = []
    attempts = 0
    while len(placed) < n_cells and attempts < n_cells * 60:
        attempts += 1
        cy, cx = rng.uniform(margin, size - margin, size=2)
        if all(np.hypot(cy - py, cx - px) > min_sep for py, px in placed):
            placed.append((cy, cx))
    for i, (cy, cx) in enumerate(placed):
        shape = classes[rng.integers(len(classes))]
        r = rng.uniform(size * 0.028, size * 0.045)
        angle = rng.uniform(0, 2 * np.pi)
        single = np.zeros_like(image)
        _draw_blob(single, cy, cx, r, shape, rng, angle)
        image += single
        this_mask = single > 0.25
        instance_mask[this_mask & (instance_mask == 0)] = i + 1
        centroids.append((cy, cx))
        labels.append(shape)
        ys, xs = np.where(this_mask)
        if len(ys):
            boxes.append((ys.min(), xs.min(), ys.max(), xs.max()))
        else:
            boxes.append((cy - r, cx - r, cy + r, cx + r))
    image = blur_image(image, gaussian_psf(psf_sigma))
    image = add_gaussian_noise(image, noise_sigma, rng)
    image = np.clip(image, 0, None)
    return dict(image=image, instance_mask=instance_mask, centroids=np.array(centroids),
                shape_labels=np.array(labels), boxes=np.array(boxes), source="SIMULATED")


def dice_coefficient(mask_a: np.ndarray, mask_b: np.ndarray) -> float:
    """Dice/F1 overlap of two binary masks: 2|A∩B| / (|A|+|B|)."""
    a, b = mask_a.astype(bool), mask_b.astype(bool)
    inter = (a & b).sum()
    denom = a.sum() + b.sum()
    return float(2 * inter / denom) if denom > 0 else 1.0


def iou_masks(mask_a: np.ndarray, mask_b: np.ndarray) -> float:
    """Intersection-over-union of two binary masks."""
    a, b = mask_a.astype(bool), mask_b.astype(bool)
    union = (a | b).sum()
    return float((a & b).sum() / union) if union > 0 else 1.0


def show_image(ax, image: np.ndarray, title: str = "", cmap: str = "gray"):
    """Show one image on a given Matplotlib axis, titled, with ticks removed."""
    ax.imshow(image, cmap=cmap)
    ax.set_title(title, fontsize=10)
    ax.set_xticks([]); ax.set_yticks([])


def show_image_grid(images, titles=None, ncols: int = 4, cmap: str = "gray", figsize_scale: float = 2.6):
    """A grid of images with Matplotlib, one subplot each, titles optional."""
    n = len(images)
    ncols = min(ncols, n)
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(figsize_scale * ncols, figsize_scale * nrows))
    axes = np.atleast_1d(axes).ravel()
    for i, img in enumerate(images):
        show_image(axes[i], img, titles[i] if titles else "", cmap=cmap)
    for ax in axes[n:]:
        ax.axis("off")
    fig.tight_layout()
    return fig, axes

---

## 1. Semantic, instance and panoptic segmentation

Four related but distinct tasks are all sometimes called "segmentation":

- **(Super-pixel) grouping** — cluster pixels by low-level similarity (Module 18's SLIC) with no notion
  of "object" at all.
- **Semantic segmentation** — assign every pixel a *class* label ("nucleus" vs. "background"), with no
  distinction between two different nuclei that both get the "nucleus" label.
- **Instance segmentation** — identify the boundaries of *individual* objects (this nucleus vs. that one),
  without necessarily labeling every background pixel.
- **Panoptic segmentation** — every pixel gets both a class label *and*, for "countable" classes, an
  instance id — the union of semantic and instance segmentation.

Most bioimage analysis cares about instance segmentation (we need to measure and track *individual* cells,
not just a blob of "cell" pixels), which is why Section 5's weighted loss — specifically about separating
touching instances of the same class — matters so much here.

## 2. From classification to dense prediction: fully convolutional networks

A classification CNN (Module 16) ends in fully-connected layers that collapse all spatial information into
one class score per image. The key insight behind the **fully convolutional network (FCN)**: replace those
final FC layers with $1\times 1$ convolutions instead. A $1\times 1$ convolution is mathematically
identical to a per-pixel fully-connected layer applied independently at every spatial location — so the
network keeps a 2-D (shrunk) output map, with one class-score vector *per spatial location*, rather than
one for the whole image. Upsampled back to the input resolution (Section 3), this gives a dense, per-pixel
class prediction in a single forward pass — the first architecture able to do end-to-end, pixel-to-pixel
semantic segmentation.

Because repeated pooling/striding shrinks the spatial resolution a great deal by the final layers (losing
fine detail), FCN **combines** the coarse, deep, semantically-rich final layer with finer, shallower
layers via "skip" connections — "where" (local, shallow) information plus "what" (global, deep) information
together give a sharper final map than either alone.

In [ ]:
# ── 2.1 Turning a fully-connected layer into an equivalent 1x1 convolution ──
# An FC layer mapping 4 input features to 3 output classes:
W_fc = rng.normal(size=(3, 4))
x_vec = rng.normal(size=4)
out_fc = W_fc @ x_vec

# The identical computation as a 1x1 convolution over a 1x1 "image" with 4 channels:
conv1x1 = nn.Conv2d(4, 3, kernel_size=1, bias=False)
with torch.no_grad():
    conv1x1.weight[:] = torch.from_numpy(W_fc.reshape(3, 4, 1, 1)).float()
x_img = torch.from_numpy(x_vec.reshape(1, 4, 1, 1)).float()
out_conv = conv1x1(x_img).detach().numpy().ravel()

print("FC layer output      :", np.round(out_fc, 4))
print("1x1 convolution output:", np.round(out_conv, 4))

🔍 **What just happened?** The two outputs match exactly — a $1\times 1$
convolution with the *same weights* as an FC layer computes the *same thing*. The only difference is that
a convolution can be applied to a spatially larger input (e.g. $32\times 32$ instead of $1\times 1$),
sliding the same per-pixel computation across every location, which is precisely how FCN turns a
whole-image classifier into a dense, per-pixel predictor without changing a single learned weight.

## 3. Transposed convolution: upsampling that is also learned

Pooling and strided convolution shrink a feature map; the decoder half of a segmentation network needs to
grow it back toward the input resolution. **Transposed convolution** ("deconvolution," or "fractionally
strided convolution") does this with *learned* weights, the same way ordinary convolution's weights are
learned — think of it as swapping convolution's forward and backward passes: where a normal convolution
maps many input pixels to one output pixel, a transposed convolution maps one input pixel to many output
pixels, using a (learned) kernel to decide how. Concretely, an ordinary $3\times3$ convolution can be
written as a matrix multiply by a sparse matrix $C$; transposed convolution with the "same" kernel is
multiplication by $C^\top$ — literally the transpose, hence the name — which maps a small feature map back
up to a larger one while preserving the same local connectivity pattern.

In [ ]:
# ── 3.1 Transposed convolution upsampling a 2x2 map back to 4x4 ──
up = nn.ConvTranspose2d(1, 1, kernel_size=3, stride=2, padding=1, output_padding=1, bias=False)
small_map = torch.tensor([[[[1.0, 2.0], [3.0, 4.0]]]])   # a 1x1x2x2 "feature map"
with torch.no_grad():
    upsampled = up(small_map)
print("input shape :", tuple(small_map.shape))
print("output shape:", tuple(upsampled.shape))
print(upsampled.detach().numpy()[0, 0].round(2))

✏️ **Your turn.** Change `stride=2` to `stride=1` in Section 3.1 (keep
`output_padding=0` to avoid a shape error) and re-run. Transposed convolution with stride 1 does *not*
change the spatial size — only `stride > 1` upsamples. This mirrors how ordinary convolution's `stride`
controls *downsampling*; transposed convolution is the same control, run in reverse.

## 4. U-Net: an encoder-decoder with skip connections

**U-Net** (Ronneberger, Fischer & Brox, 2015) was designed specifically for biomedical image segmentation,
where annotated training data is scarce. Its architecture is a symmetric "U": a **contracting path**
(repeated conv + pool, like a classification CNN's early layers, capturing *context*) followed by an
**expanding path** (repeated transposed convolution + conv, restoring resolution, enabling *precise
localization*).

U-Net's key difference from a plain FCN: at every resolution level, the expanding path **concatenates** the
corresponding contracting-path feature map (via a "skip connection") before its own convolutions, rather
than relying solely on the (lossy) upsampled coarse features. This gives the expanding path access to
high-resolution detail that would otherwise have been discarded by pooling — exactly the "where + what"
combination from Section 2, but applied repeatedly at every scale rather than just once at the end, which
is why U-Net's expanding path needs a *large* number of feature channels (to carry all that concatenated
information) and ends up roughly symmetric to the contracting path, giving the architecture its "U" shape.
U-Net also uses extensive data augmentation (including elastic deformation, simulating realistic tissue
variation) to make the most of a small annotated training set — a detail this notebook's simulator sidesteps
by simply generating unlimited synthetic training images instead.

In [ ]:
# ── 4.1 A small PyTorch U-Net ──
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1), nn.BatchNorm2d(out_ch), nn.ReLU(),
            nn.Conv2d(out_ch, out_ch, 3, padding=1), nn.BatchNorm2d(out_ch), nn.ReLU(),
        )

    def forward(self, x):
        return self.net(x)


class SmallUNet(nn.Module):
    """A compact U-Net: 2 contracting levels, a bottleneck, 2 expanding levels with skip connections."""

    def __init__(self):
        super().__init__()
        self.enc1 = ConvBlock(1, 16)
        self.enc2 = ConvBlock(16, 32)
        self.bottleneck = ConvBlock(32, 64)
        self.up2 = nn.ConvTranspose2d(64, 32, 2, stride=2)
        self.dec2 = ConvBlock(64, 32)   # 64 = 32 (skip) + 32 (upsampled)
        self.up1 = nn.ConvTranspose2d(32, 16, 2, stride=2)
        self.dec1 = ConvBlock(32, 16)   # 32 = 16 (skip) + 16 (upsampled)
        self.out_conv = nn.Conv2d(16, 1, 1)   # 1x1 conv: per-pixel foreground score (Section 2)

    def forward(self, x):
        e1 = self.enc1(x)                       # contracting path, level 1
        e2 = self.enc2(F.max_pool2d(e1, 2))      # contracting path, level 2
        b = self.bottleneck(F.max_pool2d(e2, 2)) # bottleneck
        d2 = self.dec2(torch.cat([self.up2(b), e2], dim=1))    # expanding path, with skip from e2
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))   # expanding path, with skip from e1
        return self.out_conv(d1)

unet = SmallUNet()
test_in = torch.zeros(1, 1, 64, 64)
print("output shape for a 64x64 input:", tuple(unet(test_in).shape), "(one foreground score per pixel)")

🔍 **What just happened?** `torch.cat([..., e2], dim=1)` and
`torch.cat([..., e1], dim=1)` are the skip connections — the expanding path's own upsampled features are
concatenated, channel-wise, with the matching-resolution contracting-path features *before* the decoder's
own convolutions see them. Removing those two `torch.cat` calls (replacing them with just `self.up2(b)` and
`self.up1(d2)` alone) would turn this into a plain FCN-style encoder-decoder with no skip connections —
exactly the ablation the lecture material points to as U-Net's defining improvement.

## 5. The weighted loss for touching objects

Plain per-pixel cross-entropy treats every background pixel identically — but the pixels *between* two
touching cells are far more important to get right than ordinary background, because getting them wrong
merges two cells into one false instance. U-Net's original paper addresses this with a **per-pixel weight
map** $w(x)$ that up-weights exactly those narrow background pixels sitting between nearby cells:

$$w(x) = w_c(x) + w_0 \cdot \exp\!\Big(-\frac{(d_1(x)+d_2(x))^2}{2\sigma^2}\Big)$$

where $w_c$ balances overall class frequency, and $d_1(x), d_2(x)$ are the distances from pixel $x$ to the
border of the *nearest* and *second-nearest* cell respectively — so the extra weight term is large exactly
where a pixel sits close to two different cells' borders at once (a likely touching-boundary pixel) and
near zero everywhere else.

In [ ]:
# ── 5.1 Build a border-weight map from a known instance mask ──
def border_weight_map(instance_mask, w0=10.0, sigma=5.0, class_balance_bg=1.0, class_balance_fg=1.0):
    n_instances = instance_mask.max()
    if n_instances < 2:
        w_c = np.where(instance_mask > 0, class_balance_fg, class_balance_bg)
        return w_c
    # distance, from every pixel, to the boundary of each individual instance
    dist_stack = []
    for i in range(1, n_instances + 1):
        this_instance = instance_mask == i
        dist_stack.append(ndi.distance_transform_edt(~this_instance))
    dist_stack = np.stack(dist_stack, axis=0)
    sorted_d = np.sort(dist_stack, axis=0)
    d1, d2 = sorted_d[0], sorted_d[1]
    w_c = np.where(instance_mask > 0, class_balance_fg, class_balance_bg)
    return w_c + w0 * np.exp(-((d1 + d2) ** 2) / (2 * sigma ** 2))

touching = simulate_nuclei_image(n_cells=8, size=80, touching=True, seed=7)
weight_map = border_weight_map(touching["instance_mask"])
fig, axes = plt.subplots(1, 3, figsize=(10, 3.2))
show_image(axes[0], touching["image"], "input image")
axes[1].imshow(touching["instance_mask"], cmap="nipy_spectral"); axes[1].set_title("true instance mask")
im = axes[2].imshow(weight_map, cmap="inferno"); axes[2].set_title("per-pixel loss weight")
for ax in axes[1:]: ax.set_xticks([]); ax.set_yticks([])
fig.colorbar(im, ax=axes[2], shrink=0.8)
plt.tight_layout(); plt.show()

🧪 **Ground-truth check.** Because the simulator gives us the exact instance
mask (not just a foreground/background mask), we can compute the *true* distance from every pixel to every
individual cell's boundary — something a real annotation pipeline has to approximate from manually-drawn
instance outlines. Confirm visually: the brightest streaks in the weight map should sit exactly in the thin
background gaps between pairs of touching cells, not scattered randomly across the background.

## 6. Dice loss and training a small U-Net

Per-pixel cross-entropy loss has a problem when foreground objects are small relative to the image: getting
a few foreground pixels wrong barely changes the *average* per-pixel loss, so the optimizer has little
incentive to get small objects right. The **Dice coefficient** $\text{Dice}=\dfrac{2\lvert A\cap
B\rvert}{\lvert A\rvert+\lvert B\rvert}$ (identical to the F1 score, applied to pixel masks) directly
measures the overlap fraction regardless of how small the object is relative to the image, so a
**Dice loss** ($1-\text{Dice}$) that is minimized directly rewards getting small objects' overlap right,
with **no need to manually balance foreground/background class weights** the way cross-entropy requires.
We train our U-Net with a combination: ordinary (weighted) cross-entropy for stable early training, plus a
Dice term for the small-object sensitivity.

In [ ]:
# ── 6.1 Build a training set of touching-nuclei images and instance masks ──
def make_seg_dataset(n_images, size=64, seed0=0):
    images, fg_masks, weights = [], [], []
    for k in range(n_images):
        sim = simulate_nuclei_image(n_cells=rng.integers(4, 8), size=size, touching=True,
                                     noise_sigma=rng.uniform(0.02, 0.05), seed=seed0 + k)
        images.append(sim["image"])
        fg_masks.append((sim["instance_mask"] > 0).astype(np.float32))
        weights.append(border_weight_map(sim["instance_mask"]).astype(np.float32))
    X = torch.from_numpy(np.stack(images)[:, None]).float()
    Y = torch.from_numpy(np.stack(fg_masks)[:, None]).float()
    W = torch.from_numpy(np.stack(weights)[:, None]).float()
    return X, Y, W

X_train, Y_train, W_train = make_seg_dataset(40, seed0=3000)
X_val, Y_val, W_val = make_seg_dataset(10, seed0=6000)
print("train:", X_train.shape, "val:", X_val.shape)

In [ ]:
# ── 6.2 Weighted BCE + Dice loss, and the training loop ──
def dice_loss_torch(logits, targets, eps=1e-6):
    probs = torch.sigmoid(logits)
    inter = (probs * targets).sum(dim=(1, 2, 3))
    denom = probs.sum(dim=(1, 2, 3)) + targets.sum(dim=(1, 2, 3))
    return 1.0 - ((2 * inter + eps) / (denom + eps)).mean()

model = SmallUNet()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
history = []
for epoch in range(25):
    model.train()
    perm = torch.randperm(len(X_train))
    epoch_loss = 0.0
    for i in range(0, len(X_train), 8):
        idx = perm[i:i + 8]
        optimizer.zero_grad()
        logits = model(X_train[idx])
        bce = F.binary_cross_entropy_with_logits(logits, Y_train[idx], weight=W_train[idx])
        dloss = dice_loss_torch(logits, Y_train[idx])
        loss = bce + dloss
        loss.backward(); optimizer.step()
        epoch_loss += loss.item() * len(idx)
    history.append(epoch_loss / len(X_train))

print("final training loss:", round(history[-1], 4))

In [ ]:
# ── 6.3 Evaluate on held-out validation images with Dice and IoU against ground truth ──
model.eval()
with torch.no_grad():
    val_logits = model(X_val)
    val_pred = (torch.sigmoid(val_logits) > 0.5).numpy()[:, 0]
val_true = Y_val.numpy()[:, 0] > 0.5

dices = [dice_coefficient(val_pred[i], val_true[i]) for i in range(len(val_pred))]
ious = [iou_masks(val_pred[i], val_true[i]) for i in range(len(val_pred))]
print(f"validation Dice:  mean={np.mean(dices):.3f}  (per image: {[round(d, 2) for d in dices]})")
print(f"validation IoU :  mean={np.mean(ious):.3f}")

fig, axes = plt.subplots(2, 4, figsize=(11, 5.5))
for i in range(4):
    show_image(axes[0, i], X_val[i, 0].numpy(), f"input #{i}")
    axes[1, i].imshow(val_true[i], cmap="Greens", alpha=0.6)
    axes[1, i].imshow(val_pred[i], cmap="Reds", alpha=0.4)
    axes[1, i].set_title(f"Dice={dices[i]:.2f}"); axes[1, i].set_xticks([]); axes[1, i].set_yticks([])
plt.tight_layout(); plt.show()

🔍 **What just happened?** The bottom row overlays the true foreground mask
(green) and the U-Net's predicted foreground mask (red) — pixels where they agree look olive/brown,
disagreement shows as pure green (missed) or pure red (false positive). A small model trained for only 25
quick epochs on 40 synthetic images already gets a respectable Dice score; a production model would use far
more data, elastic-deformation augmentation (as in the original U-Net paper), and typically far more
capacity.

✏️ **Your turn.** Retrain with `W_train` replaced by `torch.ones_like(W_train)`
everywhere (i.e. remove the border weighting, keeping only plain unweighted BCE + Dice) and compare the
resulting Dice/IoU scores, especially by eye on images with several touching cells. Does the model without
border weighting produce more "fused" blobs where two true cells touch?

### 📘 Lecture notes — V-Net and volumetric segmentation

**V-Net** (Milletari, Navab & Ahmadi, 2016) applies the same encoder-decoder idea to full 3-D volumes
(e.g. MRI stacks) using 3-D convolutions throughout, and was the paper that introduced the Dice loss in
exactly this role — directly optimizing the overlap metric used to *evaluate* segmentation, rather than a
proxy loss like cross-entropy, and sidestepping the need to hand-tune foreground/background class weights
that pure cross-entropy requires for small structures. Module 33 returns to the volumetric case at
production scale.

---

## 7. Flow-based instance segmentation (Cellpose-style)

Sections 5–6 separated touching objects by shaping the **loss** (a border weight map) around an otherwise
ordinary semantic U-Net; the instances themselves still come from running connected components on the
output mask after training. That means the method can never do better, at instance separation, than
whatever gap is left in the thresholded mask — if two true objects' predicted masks end up touching with
*no* gap, connected components is mathematically forced to label them as one object, however good the
rest of the network is.

**Cellpose** (Stringer et al., 2021) sidesteps this by changing what the network predicts, not the loss.
Instead of a foreground/background mask, the network predicts, at every foreground pixel, a 2-D vector
pointing toward the **centre of that pixel's own object** — plus a foreground/background logit, in a
second output channel, trained jointly. At inference time you do not threshold and label; you let every
foreground pixel *walk* along its own predicted vector, a few pixels at a time, and watch where it ends up.
Pixels belonging to the same object converge to (almost) the same point — a "sink" — regardless of whether
their silhouettes touch, because the vector field itself already encodes which object each pixel belongs to.
Instances are recovered by clustering the sinks, not the original pixels.

> 📘 **A deliberate simplification.** Real Cellpose derives its target vector field from the *gradient of a
> simulated heat-diffusion process* seeded at each object's centre (closer in spirit to the medial axis than
> to a straight line), which behaves better on concave and elongated shapes. Here we use the simplest
> possible ground-truth field a simulator can give for free — the straight-line unit vector from each
> foreground pixel to its own object's true centroid — to keep the idea itself front and centre. The
> decoding step (follow the field, then cluster the sinks) is the same algorithm Cellpose actually uses.


In [ ]:
# ── 7.1 Ground-truth flow field: at each foreground pixel, the unit vector toward its own
#      instance's centroid (background = 0) ──
def make_flow_field(instance_mask, centroids):
    size = instance_mask.shape[0]
    yy, xx = np.meshgrid(np.arange(size), np.arange(size), indexing='ij')
    flow = np.zeros((2, size, size), dtype=np.float32)
    for i, (cy, cx) in enumerate(centroids, start=1):
        m = instance_mask == i
        dy, dx = cy - yy[m], cx - xx[m]
        norm = np.hypot(dy, dx) + 1e-6
        flow[0][m] = dy / norm
        flow[1][m] = dx / norm
    return flow

sim_demo = simulate_nuclei_image(n_cells=10, size=64, touching=True, seed=SEED + 6999)
flow_demo = make_flow_field(sim_demo['instance_mask'], sim_demo['centroids'])

fig, axes = plt.subplots(1, 3, figsize=(10, 3.4))
show_image(axes[0], sim_demo['image'], 'input (SIMULATED)')
axes[1].imshow(sim_demo['instance_mask'], cmap='tab20'); axes[1].set_title('true instances')
axes[1].set_xticks([]); axes[1].set_yticks([])
step = 4
yy, xx = np.meshgrid(np.arange(64), np.arange(64), indexing='ij')
fg = sim_demo['instance_mask'] > 0
axes[2].imshow(sim_demo['instance_mask'] > 0, cmap='Greys', alpha=0.3)
axes[2].quiver(xx[::step, ::step], yy[::step, ::step],
               flow_demo[1, ::step, ::step], -flow_demo[0, ::step, ::step],
               np.where(fg[::step, ::step], 1, 0), cmap='viridis', scale=30)
axes[2].set_title('ground-truth flow (→ own centroid)'); axes[2].set_xticks([]); axes[2].set_yticks([])
axes[2].invert_yaxis()
plt.tight_layout(); plt.show()


🔍 **What just happened?** Every arrow points from a foreground pixel toward the centroid of *its own*
object, computed directly from the simulator's instance mask — this is the training target, available only
because we generated the data ourselves. Notice that a pixel right at the touching boundary between two
cells has a flow vector pointing toward *its own* cell's centre and nowhere near its neighbour's — this is
exactly the information a thresholded mask throws away, and exactly what lets the decoding step (Section 7.4)
separate the two cells again.


In [ ]:
# ── 7.2 A joint flow + foreground network: same compact encoder-decoder shape as SmallUNet
#      (Section 4.1), but 3 output channels — 2 flow components and 1 foreground logit ──
class FlowUNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc1 = ConvBlock(1, 16)
        self.enc2 = ConvBlock(16, 32)
        self.bottleneck = ConvBlock(32, 64)
        self.up2 = nn.ConvTranspose2d(64, 32, 2, stride=2)
        self.dec2 = ConvBlock(64, 32)
        self.up1 = nn.ConvTranspose2d(32, 16, 2, stride=2)
        self.dec1 = ConvBlock(32, 16)
        self.out_conv = nn.Conv2d(16, 3, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(F.max_pool2d(e1, 2))
        b = self.bottleneck(F.max_pool2d(e2, 2))
        d2 = self.dec2(torch.cat([self.up2(b), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))
        out = self.out_conv(d1)
        return out[:, :2], out[:, 2:3]  # (flow, foreground logit)


In [ ]:
# ── 7.3 Build a flow-field training set (dense, 10-cell images so touching/merged
#      instances are common) and train FlowUNet ──
def make_flow_dataset(n_images, size=64, n_cells=10, seed0=0):
    Xs, Fs, Ms = [], [], []
    for i in range(n_images):
        sim = simulate_nuclei_image(n_cells=n_cells, size=size, touching=True, seed=seed0 + i)
        Xs.append(sim['image'])
        Fs.append(make_flow_field(sim['instance_mask'], sim['centroids']))
        Ms.append((sim['instance_mask'] > 0).astype(np.float32))
    X = torch.from_numpy(np.stack(Xs)).float().unsqueeze(1)
    Fgt = torch.from_numpy(np.stack(Fs)).float()
    M = torch.from_numpy(np.stack(Ms)).float().unsqueeze(1)
    return X, Fgt, M

X_flow, F_flow, M_flow = make_flow_dataset(30, size=64, n_cells=10, seed0=SEED + 7000)

flow_net = FlowUNet()
flow_opt = torch.optim.Adam(flow_net.parameters(), lr=1e-3)
for epoch in range(600):
    flow_net.train()
    pred_flow, pred_fg_logit = flow_net(X_flow)
    flow_loss = ((pred_flow - F_flow) ** 2 * M_flow).sum() / M_flow.sum().clamp(min=1)
    fg_loss = F.binary_cross_entropy_with_logits(pred_fg_logit, M_flow)
    loss = flow_loss + fg_loss
    flow_opt.zero_grad(); loss.backward(); flow_opt.step()

print('final training loss:', round(loss.item(), 4),
      '(flow MSE on foreground + foreground BCE)')


In [ ]:
# ── 7.4 Decode instances by following the predicted flow to convergence, then clustering
#      the resulting sink points — the same two-stage algorithm Cellpose itself uses ──
def follow_flow(flow, fg_mask, n_steps=60, step_size=1.0):
    size = flow.shape[1]
    yy, xx = np.meshgrid(np.arange(size), np.arange(size), indexing='ij')
    ys, xs = yy[fg_mask].astype(np.float64), xx[fg_mask].astype(np.float64)
    fy, fx = flow[0], flow[1]
    for _ in range(n_steps):
        iy = np.clip(ys.round().astype(int), 0, size - 1)
        ix = np.clip(xs.round().astype(int), 0, size - 1)
        ys = np.clip(ys + step_size * fy[iy, ix], 0, size - 1)
        xs = np.clip(xs + step_size * fx[iy, ix], 0, size - 1)
    return ys, xs

def cluster_sinks(ys, xs, size, grid=1.0, dil_iters=1):
    """Rasterize converged sink points to a fine grid, dilate slightly to merge sinks from the
    same object, then connected-component label — pixels are assigned the component their own sink
    landed in."""
    grid_size = int(size / grid) + 1
    occ = np.zeros((grid_size, grid_size), dtype=bool)
    gy = (ys / grid).round().astype(int).clip(0, grid_size - 1)
    gx = (xs / grid).round().astype(int).clip(0, grid_size - 1)
    occ[gy, gx] = True
    occ = ndi.binary_dilation(occ, iterations=dil_iters)
    labeled, n = ndi.label(occ)
    return labeled[gy, gx], n

def decode_instances(image, net, min_area=4):
    x = torch.from_numpy(image).float()[None, None]
    flow_net.eval()
    with torch.no_grad():
        pred_flow, pred_fg_logit = net(x)
    pred_flow = pred_flow[0].numpy()
    fg_mask = torch.sigmoid(pred_fg_logit)[0, 0].numpy() > 0.5
    ys, xs = follow_flow(pred_flow, fg_mask)
    labels_flat, _ = cluster_sinks(ys, xs, size=image.shape[0])
    size = image.shape[0]
    yy, xx = np.meshgrid(np.arange(size), np.arange(size), indexing='ij')
    pred_mask = np.zeros((size, size), dtype=int)
    pred_mask[yy[fg_mask], xx[fg_mask]] = labels_flat
    for lbl in np.unique(pred_mask):
        if lbl != 0 and (pred_mask == lbl).sum() < min_area:
            pred_mask[pred_mask == lbl] = 0
    return pred_mask, fg_mask

sim_show = simulate_nuclei_image(n_cells=10, size=64, touching=True, seed=SEED + 7777)
pred_mask_show, fg_mask_show = decode_instances(sim_show['image'], flow_net)
naive_show, _ = ndi.label(fg_mask_show)

fig, axes = plt.subplots(1, 3, figsize=(10, 3.4))
show_image(axes[0], sim_show['image'], 'input (SIMULATED)')
axes[1].imshow(naive_show, cmap='tab20'); axes[1].set_title('naive: threshold + connected components')
axes[1].set_xticks([]); axes[1].set_yticks([])
axes[2].imshow(pred_mask_show, cmap='tab20'); axes[2].set_title('flow-based decoding')
axes[2].set_xticks([]); axes[2].set_yticks([])
plt.tight_layout(); plt.show()


In [ ]:
# ── 7.5 Ground-truth check: on images with >=1 true touching pair, connected components on
#      the predicted mask is FORCED to merge them into one label (it cannot do otherwise); does
#      flow-based decoding recover the correct count and assignment? ──
def evaluate_instances(sim, pred_mask):
    true_mask = sim['instance_mask']
    true_ids = [i for i in np.unique(true_mask) if i != 0]
    correct = 0
    for t in true_ids:
        region = true_mask == t
        votes = pred_mask[region]; votes = votes[votes != 0]
        if len(votes) == 0:
            continue
        vals, counts = np.unique(votes, return_counts=True)
        best = vals[np.argmax(counts)]
        back = true_mask[pred_mask == best]; back = back[back != 0]
        vals2, counts2 = np.unique(back, return_counts=True)
        if vals2[np.argmax(counts2)] == t and counts[np.argmax(counts)] / region.sum() > 0.5:
            correct += 1
    return len(true_ids), correct

N_TEST = 40
tot_true = tot_flow_correct = tot_naive_correct = 0
n_blobs = n_blobs_flow_correct = 0

for i in range(N_TEST):
    sim = simulate_nuclei_image(n_cells=10, size=64, touching=True, seed=SEED + 8000 + i)
    true_mask = sim['instance_mask']
    pred_mask, fg_mask = decode_instances(sim['image'], flow_net)
    naive_mask, n_naive = ndi.label(fg_mask)
    for lbl in range(1, n_naive + 1):
        if (naive_mask == lbl).sum() < 4:
            naive_mask[naive_mask == lbl] = 0

    n_true, n_correct_flow = evaluate_instances(sim, pred_mask)
    _, n_correct_naive = evaluate_instances(sim, naive_mask)
    tot_true += n_true; tot_flow_correct += n_correct_flow; tot_naive_correct += n_correct_naive

    # genuinely merged blobs: one naive connected component spanning >= 2 true instances —
    # by construction, connected components alone can never report more than 1 label here
    for lbl in range(1, naive_mask.max() + 1):
        blob = naive_mask == lbl
        true_in_blob = [t for t in np.unique(true_mask[blob]) if t != 0]
        if len(true_in_blob) < 2:
            continue
        n_blobs += 1
        flow_labels = [l for l in np.unique(pred_mask[blob]) if l != 0]
        if len(flow_labels) >= len(true_in_blob):
            majority = []
            for t in true_in_blob:
                votes = pred_mask[(true_mask == t) & blob]; votes = votes[votes != 0]
                if len(votes):
                    vals, counts = np.unique(votes, return_counts=True)
                    majority.append(vals[np.argmax(counts)])
            if len(set(majority)) == len(true_in_blob):
                n_blobs_flow_correct += 1

print(f'Whole-image instance accuracy over {N_TEST} held-out images ({tot_true} true instances):')
print(f'  flow-based decoding:            {tot_flow_correct}/{tot_true}  ({100*tot_flow_correct/tot_true:.1f}%)')
print(f'  naive (threshold + conn.comp.): {tot_naive_correct}/{tot_true}  ({100*tot_naive_correct/tot_true:.1f}%)')
print(f'\nOn the {n_blobs} genuinely touching/merged true-instance blobs found above:')
print(f'  flow-based decoding recovers the correct split: {n_blobs_flow_correct}/{n_blobs}'
      f'  ({100*n_blobs_flow_correct/max(n_blobs,1):.1f}%)')
print(f'  naive (threshold + conn.comp.):                 0/{n_blobs}  (0.0% -- structurally impossible)')


🔍 **What just happened?** Averaged over whole images, flow-based decoding is only a little ahead of the
naive baseline (most cells in a random layout never actually touch, so the two methods agree there by
default). The second number is the honest, targeted one: restricted to the blobs where a true touching pair
really did get fused into one connected component in the predicted mask, naive is wrong on *every single
one* — not by bad luck, but because one connected component can only ever produce one label. Flow-based
decoding, with no access to anything connected components doesn't already have, recovers the correct split
on a real, non-trivial minority of exactly these cases — not all of them, but meaningfully more than the
zero that is all connected components can ever manage here. This is a small network trained very briefly on
a simplified, straight-line flow target, so a fair number of sinks still land close enough together to get
re-merged by the clustering step; a longer-trained model with the real heat-diffusion target would recover
more of them. The honest result demonstrates the real mechanism, not a cherry-picked one: the flow field encodes instance
identity *before* any thresholding happens, which is information a mask-then-label pipeline has already
discarded by the time it tries to separate touching objects.


✏️ **Your turn.** `cluster_sinks` has two parameters that control how aggressively nearby sinks are
merged: `grid` (the rasterization resolution) and `dil_iters` (how much the occupied grid is dilated before
labeling). Try `grid=2.0, dil_iters=2` and re-run Section 7.5 — what happens to the genuinely-merged-blob
accuracy, and why? Then try `dil_iters=0` — does it ever *over*-segment a single object into two (one true
instance whose own sink points spread across more than one grid cell)?


### 📘 Lecture notes — StarDist and the shape-representation alternative

A different way to escape connected components' limits is to change *what shape* the network is allowed to
predict, rather than predicting a vector field. **StarDist** (Schmidt et al., 2018; Weigert et al., 2020)
has each foreground pixel predict the distance to its object's boundary along a fixed number of rays from
that pixel — a star-convex polygon — plus an objectness score; instances are then recovered by non-maximum
suppression over these candidate polygons, never by labeling connected components at all. StarDist handles
round-ish, non-overlapping objects (most nuclei) particularly well and is a common first choice in practice;
Cellpose's flow-field approach is more general-purpose for the elongated, concave or heavily touching shapes
where a star-convex polygon is a poor fit. Both replace *where does a boundary happen to leave a gap?* with
*what does this pixel itself believe about the object it belongs to?* — the same shift in kind as the
weighted loss of Section 5, pushed one step further into the network's output representation itself.


📌 **Key takeaways**

- Semantic segmentation labels pixels by class; instance segmentation additionally distinguishes individual objects of the same class; panoptic segmentation does both at once.
- FCN replaces a classifier's final FC layers with $1\times1$ convolutions, giving a dense, per-pixel prediction in one forward pass; transposed convolution learns how to upsample those coarse predictions back toward the input resolution.
- U-Net's defining feature beyond a plain FCN is its skip connections, concatenating matching-resolution encoder features into the decoder at every scale, not just at the end — giving fine localization *and* global context simultaneously.
- A per-pixel weight map that up-weights the narrow gaps between touching objects directly addresses instance segmentation's hardest failure mode: merged, touching objects.
- Dice loss optimizes overlap directly, which particularly helps when foreground objects are small relative to the image, without needing to hand-balance class weights the way cross-entropy does.
- Flow-based (Cellpose-style) instance segmentation moves the separation of touching objects out of the loss and into the network's output itself: a per-pixel vector field toward each object's own centre, decoded by following the field to convergence and clustering the sinks — which can split objects that thresholding-then-connected-components structurally cannot.


✅ **Checkpoint**

1. What, specifically, would you remove from the `SmallUNet` code in Section 4.1 to turn it into a plain
   encoder-decoder (no skip connections), and what would you expect to happen to segmentation sharpness as
   a result?
   <details><summary>Answer</summary>Remove the `torch.cat([..., e2], ...)` and `torch.cat([..., e1],
   ...)` concatenations (using just the upsampled features alone); you would expect coarser, less precisely
   localized boundaries, since the decoder would lose access to high-resolution detail from the encoder.
   </details>
2. Why is a $1\times1$ convolution the natural replacement for a fully-connected layer in a dense
   prediction network?
   <details><summary>Answer</summary>Because, applied at a single spatial location, it performs *exactly*
   the same weighted-sum computation as an FC layer (Section 2.1), but can be applied identically — with
   shared weights — at every spatial location of a larger feature map, preserving spatial structure instead
   of collapsing it.</details>
3. Why does a plain (unweighted) Dice loss already partly address class imbalance, without needing an
   explicit per-class weight the way cross-entropy does?
   <details><summary>Answer</summary>Dice directly measures the *overlap fraction* of predicted and true
   foreground, which is naturally sensitive to foreground pixels regardless of how small a fraction of the
   whole image they make up — unlike per-pixel cross-entropy's simple average, which a small foreground
   region barely moves.</details>
4. In Section 7.5, why is naive (threshold + connected components) guaranteed to score 0% on the
   "genuinely merged blobs" specifically — not just unlucky, but mathematically guaranteed?
   <details><summary>Answer</summary>By definition, a "genuinely merged blob" is one connected component
   in the predicted foreground mask that overlaps two or more true instances. Connected-component labeling
   assigns exactly one label per connected component, with no way to split it further — so any true
   instance pair sharing a blob can never both be recovered correctly by that method alone, regardless of
   how well the rest of the pipeline is trained.</details>


🧠 **Questions & discussion**

- The original U-Net paper relied on extensive elastic-deformation data augmentation because annotated biomedical images were scarce. Our notebook instead generates unlimited synthetic training data. What kinds of real-world variation (imaging artifacts, staining differences, focus drift) would synthetic data like ours still fail to capture?
- The weighted-loss approach in Section 5 explicitly encodes the idea of 'touching objects' into the loss function, rather than the network architecture. Can you think of an architectural change (rather than a loss change) that might help separate touching instances instead?
- We combined BCE and Dice loss with equal weight (`bce + dloss`). What would you expect to happen to training stability if you used *only* Dice loss from the very first epoch, before the network has learned anything useful at all?

🚀 **Bonus — going further**

- Add a third contracting/expanding level to `SmallUNet` (go from 2 levels to 3) and see whether validation Dice improves on more cells per image.
- Increase `n_cells` in Section 7.3's training set from 10 to 16 (denser, more touching pairs) and retrain — does the genuinely-merged-blob accuracy in Section 7.5 go up, down, or stay about the same, and can you explain why?
- Replace the border-weight map's exponential with a hard binary mask (weight 10 within 3 pixels of a second-nearest-instance boundary, weight 1 elsewhere) and compare training behavior to the smooth Gaussian version.
- Replace `make_flow_field`'s straight-line-to-centroid target with a field derived from the distance transform's gradient (`scipy.ndimage.distance_transform_edt`, then `np.gradient` on the per-instance distance map) — does decoding behave differently on elongated or kidney-shaped cells, where the straight line to the centroid can leave the object's own silhouette?


📝 **Lab note**

```text
Module 19 — Deep learning segmentation
Date: ____________
U-Net architecture: enc(___, ___) -> bottleneck(___) -> dec(___, ___)
Validation Dice (mean): ____   Validation IoU (mean): ____
With border weighting vs. without -- visually fewer fused touching cells?  yes / no
One change I would try next to improve small/touching-cell separation: ____________
```

📚 **References**

- Long, J., Shelhamer, E. & Darrell, T. (2015). Fully convolutional networks for semantic segmentation. *CVPR 2015*.
- Ronneberger, O., Fischer, P. & Brox, T. (2015). U-Net: Convolutional networks for biomedical image segmentation. *MICCAI 2015*.
- Milletari, F., Navab, N. & Ahmadi, S.-A. (2016). V-Net: Fully convolutional neural networks for volumetric medical image segmentation. *3DV 2016*.
- Kirillov, A., He, K., Girshick, R., Rother, C. & Dollár, P. (2019). Panoptic segmentation. *CVPR 2019*.
- Dumoulin, V. & Visin, F. (2016). A guide to convolution arithmetic for deep learning. *arXiv:1603.07285*.
- Stringer, C., Wang, T., Michaelos, M. & Pachitariu, M. (2021). Cellpose: a generalist algorithm for cellular segmentation. *Nature Methods*, 18(1), 100–106.
- Schmidt, U., Weigert, M., Broaddus, C. & Myers, G. (2018). Cell detection with star-convex polygons. *MICCAI 2018*.
- Weigert, M., Schmidt, U., Haase, R., Sugawara, K. & Myers, G. (2020). Star-convex polyhedra for 3D object detection and segmentation in microscopy. *WACV 2020*.


➡️ **Next.** Module 20 moves from *which pixels* to *where are the objects and what are they*: sliding-window detection, anchors, non-maximum suppression, and the YOLO/SSD/RetinaNet/R-CNN families of object detectors.